# 03 — Recommender Model

Build the TF-IDF + cosine similarity content-based recommender (`src/recommender.py`) and demo it on sample products.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.recommender import FashionRecommender

df = pd.read_csv("../data/processed/cleaned_fashion_products.csv")
df.head()

,User ID,Product ID,Product Name,Brand,Category,Price,Rating,Color,Size,content_profile,Price_scaled,Rating_scaled
0,19,1,Dress,Adidas,Men's Fashion,40,1.043159,Black,XL,Dress Adidas Men's Fashion Black XL,0.333333,0.010582
1,97,2,Shoes,H&M,Women's Fashion,82,4.026416,Black,L,Shoes H&M Women's Fashion Black L,0.800000,0.758829
2,25,3,Dress,Adidas,Women's Fashion,44,3.337938,Yellow,XL,Dress Adidas Women's Fashion Yellow XL,0.377778,0.586148
3,57,4,Shoes,Zara,Men's Fashion,23,1.049523,White,S,Shoes Zara Men's Fashion White S,0.144444,0.012179
4,79,5,T-shirt,Adidas,Men's Fashion,79,4.302773,Black,M,T-shirt Adidas Men's Fashion Black M,0.766667,0.828144


## Fit the recommender

TF-IDF vectorizes each product's `content_profile`, then Price/Rating (scaled) are appended as extra weighted numeric columns before computing the full pairwise cosine similarity matrix.

In [2]:
model = FashionRecommender(numeric_weight=0.5).fit(df)
print("Similarity matrix shape:", model.similarity_matrix.shape)

Similarity matrix shape: (1000, 1000)


## Demo: recommend similar items

In [3]:
sample_product_id = int(df["Product ID"].iloc[1])
query = df[df["Product ID"] == sample_product_id][["Product ID", "Product Name", "Brand", "Category", "Color", "Price", "Rating"]]
print("Query product:")
display(query)

recommendations = model.recommend(sample_product_id, top_n=5)
print("\nTop 5 recommendations:")
recommendations

Query product:


,Product ID,Product Name,Brand,Category,Color,Price,Rating
1,2,Shoes,H&M,Women's Fashion,Black,82,4.026416



Top 5 recommendations:


,Product ID,Product Name,Brand,Category,Color,Price,Rating,similarity_score
0,377,Shoes,Zara,Women's Fashion,Black,54,3.722545,0.881964
1,148,Shoes,Adidas,Women's Fashion,Black,82,2.716312,0.880688
2,160,Shoes,Gucci,Women's Fashion,Black,35,2.983803,0.850475
3,557,Shoes,Adidas,Women's Fashion,Black,85,1.502022,0.850114
4,749,Shoes,Gucci,Women's Fashion,Black,43,2.386806,0.848341


In [4]:
for pid in [2, 500, 750]:
    print(f"=== Recommendations for Product ID {pid} ===")
    display(model.recommend(pid, top_n=5))
    print()

=== Recommendations for Product ID 2 ===


,Product ID,Product Name,Brand,Category,Color,Price,Rating,similarity_score
0,377,Shoes,Zara,Women's Fashion,Black,54,3.722545,0.881964
1,148,Shoes,Adidas,Women's Fashion,Black,82,2.716312,0.880688
2,160,Shoes,Gucci,Women's Fashion,Black,35,2.983803,0.850475
3,557,Shoes,Adidas,Women's Fashion,Black,85,1.502022,0.850114
4,749,Shoes,Gucci,Women's Fashion,Black,43,2.386806,0.848341



=== Recommendations for Product ID 500 ===


,Product ID,Product Name,Brand,Category,Color,Price,Rating,similarity_score
0,546,Shoes,Zara,Women's Fashion,Red,93,1.356262,0.998920
1,298,Shoes,Zara,Women's Fashion,Red,14,1.416317,0.926090
2,417,Shoes,H&M,Women's Fashion,Red,88,2.846892,0.881605
3,461,Shoes,Zara,Kids' Fashion,Red,93,1.030127,0.861871
4,840,Shoes,Zara,Kids' Fashion,Red,84,3.154721,0.853143



=== Recommendations for Product ID 750 ===


,Product ID,Product Name,Brand,Category,Color,Price,Rating,similarity_score
0,233,Shoes,Nike,Kids' Fashion,Red,39,4.706320,0.989943
1,843,Shoes,Nike,Kids' Fashion,Red,91,2.723013,0.979162
2,991,Shoes,Nike,Kids' Fashion,Red,25,4.972677,0.977458
3,409,Shoes,Nike,Kids' Fashion,Red,26,2.010899,0.968172
4,893,Shoes,H&M,Kids' Fashion,Red,76,2.072748,0.864500


The top recommendations consistently match on product type and color first, then rank by closeness in brand, price, and rating — confirming the blended TF-IDF + numeric cosine similarity approach captures both categorical and continuous similarity.

## Save the fitted model artifacts

Persisted so the Streamlit app doesn't need to refit on every run.

In [5]:
model.save("../outputs/recommender_artifacts.pkl")
print("Saved model artifacts to ../outputs/recommender_artifacts.pkl")

Saved model artifacts to ../outputs/recommender_artifacts.pkl


## Feedback-driven learning

The engine isn't static: `record_feedback(query_id, rec_id, signal)` nudges a persistent item-item boost that's added to the base cosine similarity score on every future `recommend()` call for that pair, and is saved to disk so it survives restarts. Below, we simulate a user repeatedly disliking the #1 recommendation for a product and show the ranking shift.

In [6]:
demo_product_id = int(df["Product ID"].iloc[1])

print("=== Before feedback ===")
before = model.recommend(demo_product_id, top_n=5)
display(before[["Product ID", "Product Name", "Brand", "similarity_score"]])

disliked_id = int(before.iloc[0]["Product ID"])
print(f"Simulating 3x dislike on top result (Product ID {disliked_id})...")
for _ in range(3):
    new_boost = model.record_feedback(demo_product_id, disliked_id, signal=-1, log_path="../outputs/feedback_log.csv")
print("New boost for this pair:", round(new_boost, 4))

=== Before feedback ===


,Product ID,Product Name,Brand,similarity_score
0,377,Shoes,Zara,0.881964
1,148,Shoes,Adidas,0.880688
2,160,Shoes,Gucci,0.850475
3,557,Shoes,Adidas,0.850114
4,749,Shoes,Gucci,0.848341


Simulating 3x dislike on top result (Product ID 377)...
New boost for this pair: -0.15


In [7]:
print("=== After feedback ===")
after = model.recommend(demo_product_id, top_n=5)
display(after[["Product ID", "Product Name", "Brand", "similarity_score"]])

print()
print(f"Product ID {disliked_id} still in top 5?", disliked_id in after["Product ID"].values)

=== After feedback ===


,Product ID,Product Name,Brand,similarity_score
0,148,Shoes,Adidas,0.880688
1,160,Shoes,Gucci,0.850475
2,557,Shoes,Adidas,0.850114
3,749,Shoes,Gucci,0.848341
4,647,Shoes,Adidas,0.844220



Product ID 377 still in top 5? False


The disliked item drops out of (or down in) the ranking, while everything else's relative order is unaffected — confirming the boost is applied precisely to the pair that received feedback, not globally. This is what "improves with usage" means concretely here: a real, persistent, inspectable state change driven by user interaction, not a simulated metric.

In [8]:
# Re-save so the persisted artifacts include this demo's feedback state
model.save("../outputs/recommender_artifacts.pkl")
print("Saved model (with feedback_boost) to ../outputs/recommender_artifacts.pkl")

Saved model (with feedback_boost) to ../outputs/recommender_artifacts.pkl
